In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import hnsw_benchmark2 as bm
import farhest_point_order as fp
from datasets import load_dataset
import os

In [ ]:
# Alapadatok megadása
dataset_name = "arguana"
embedding_model = "e5-small-v2"

calc_lid = True
calc_fps = True

In [ ]:
# Adathalmazok beolvasása
# Corpus
corpus = load_dataset("mteb/" + dataset_name, "corpus")
corpus_texts = corpus.data["corpus"]["text"]
print(f"Corpus size: {len(corpus_texts):10d}")

# Query
queries = load_dataset("mteb/" + dataset_name, "queries")
query_texts = queries.data["queries"]["text"]
print(f"Query size:  {len(corpus_texts):10d}")

In [ ]:
print(corpus_texts[0])
# print(query_texts[0])

In [ ]:
# Embedding végrehajtása
model = SentenceTransformer("intfloat/" + embedding_model)

corpus_embeddings = model.encode(
    [f"passage: {text}" for text in corpus_texts],
    normalize_embeddings=True,
    batch_size=64,
    show_progress_bar=True,
)

query_embeddings = model.encode(
    [f"query: {text}" for text in query_texts],
    normalize_embeddings=True,
    batch_size=64,
    show_progress_bar=True,
)


corpus_embeddings = np.asarray(corpus_embeddings, dtype=np.float32)
query_embeddings = np.asarray(query_embeddings, dtype=np.float32)

In [ ]:
print(f"Corpus dimensions: {corpus_embeddings.shape}")
print(f"Query dimensions:  {query_embeddings.shape}")

In [ ]:
# LID számolás
if calc_lid:
    corpus_lids = bm.calculate_lid(corpus_embeddings)

In [ ]:
# Farthest points igazi számítása
if calc_fps:
    corpus_farthest_scores = fp.farthest_point_order(corpus_embeddings)

In [ ]:
# Fájlok kiírása
# Corpus
df_corpus = pd.DataFrame(columns=["vectors"], data=[[v] for v in corpus_embeddings])
corpus_filename = dataset_name + "_corpus_" + embedding_model

if calc_lid:
    df_corpus.assign(LID=corpus_lids)
    corpus_filename = corpus_filename + "_lid"
if calc_fps:
    df_corpus.assign(fp_score=corpus_farthest_scores)
    corpus_filename = corpus_filename + "_fps"

corpus_output_path = os.path.join("..\\data\\", corpus_filename + ".json")
df_corpus.to_json(corpus_output_path, orient="records", lines=True)


# Query
df_query = pd.DataFrame(columns=["vectors"], data=[[v] for v in query_embeddings])
query_filename = dataset_name + "_query_" + embedding_model
query_output_path = os.path.join("..\\data\\", query_filename + ".json")
df_query.to_json(query_output_path, orient="records", lines=True)